# **<u>EXPERIMENT-6: GloVe Word Embeddings and Language Modelling</u>**

## **Experiment 6.1:** To implement the GloVe (Global Vectors for Word Representation) algorithm to generate word embeddings from the text corpus and find most similar words of “King”.


(Two separate codes 1. Without using any pretrained model 2. Using Pretrained model (Gensim) )

Dataset link:

```https://drive.google.com/file/d/1-5tQzT-rVkiwp_eVOgU9mT4U_v-FIlFH/view?usp=drive_link```

## a. Without using any pretrained model

In [1]:
import numpy as np
import re
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# 1. Load corpus

with open("glove_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

# Remove punctuation and tokenize
text = re.sub(r"[^a-z\s]", "", text)
tokens = text.split()

print("Total tokens:", len(tokens))

Total tokens: 148


In [3]:
# 2. Build vocabulary
word_counts = Counter(tokens)

vocab = sorted(word_counts.keys())
word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)
print("Vocabulary:")
print(vocab)

Vocabulary size: 68
Vocabulary:
['a', 'and', 'around', 'at', 'books', 'brave', 'cat', 'computer', 'conducts', 'data', 'daughter', 'develops', 'doctor', 'dog', 'drinks', 'earth', 'eats', 'food', 'gives', 'heat', 'house', 'in', 'information', 'intelligent', 'is', 'kind', 'king', 'kingdom', 'laboratory', 'learn', 'light', 'live', 'lives', 'man', 'milk', 'moon', 'new', 'night', 'of', 'palace', 'patient', 'powerful', 'prince', 'princess', 'processes', 'queen', 'read', 'research', 'revolves', 'royal', 'rule', 'rules', 'school', 'scientist', 'shines', 'software', 'son', 'strong', 'students', 'sun', 'teacher', 'teaches', 'the', 'things', 'together', 'treats', 'wise', 'woman']


In [4]:
# 3. Construct co-occurrence matrix

window_size = 3

cooccurrence = np.zeros(
    (vocab_size, vocab_size),
    dtype=np.float64
)

for i, word in enumerate(tokens):

    center_id = word_to_id[word]

    left = max(0, i - window_size)
    right = min(len(tokens), i + window_size + 1)

    for j in range(left, right):

        if i == j:
            continue

        context_word = tokens[j]
        context_id = word_to_id[context_word]

        # Distance weighting
        distance = abs(i - j)
        cooccurrence[center_id][context_id] += 1.0 / distance

print("\nCo-occurrence matrix shape:",
      cooccurrence.shape)


Co-occurrence matrix shape: (68, 68)


In [5]:
# 4. GloVe parameters

embedding_dim = 50
learning_rate = 0.05
epochs = 500
x_max = 100
alpha = 0.75

np.random.seed(42)

W = np.random.uniform(
    -0.5 / embedding_dim,
    0.5 / embedding_dim,
    (vocab_size, embedding_dim)
)

W_context = np.random.uniform(
    -0.5 / embedding_dim,
    0.5 / embedding_dim,
    (vocab_size, embedding_dim)
)

b = np.zeros(vocab_size)
b_context = np.zeros(vocab_size)

In [6]:
# 5. GloVe weighting function

def weighting_function(x):
    if x < x_max:
        return (x / x_max) ** alpha
    return 1.0

In [7]:
# 6. Train GloVe

non_zero = np.argwhere(cooccurrence > 0)

for epoch in range(epochs):

    total_loss = 0

    np.random.shuffle(non_zero)

    for i, j in non_zero:

        x_ij = cooccurrence[i, j]

        weight = weighting_function(x_ij)

        prediction = (
            np.dot(W[i], W_context[j])
            + b[i]
            + b_context[j]
        )

        error = prediction - np.log(x_ij)

        total_loss += weight * (error ** 2)

        gradient = weight * error

        # Save vectors before updating
        wi = W[i].copy()
        wj = W_context[j].copy()

        W[i] -= learning_rate * gradient * wj
        W_context[j] -= learning_rate * gradient * wi

        b[i] -= learning_rate * gradient
        b_context[j] -= learning_rate * gradient

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch+1}/{epochs}, "
            f"Loss = {total_loss:.4f}"
        )

# Final embeddings
embeddings = W + W_context

Epoch 50/500, Loss = 6.0269
Epoch 100/500, Loss = 5.4199
Epoch 150/500, Loss = 5.0547
Epoch 200/500, Loss = 4.6115
Epoch 250/500, Loss = 3.9600
Epoch 300/500, Loss = 3.3527
Epoch 350/500, Loss = 2.8767
Epoch 400/500, Loss = 2.5205
Epoch 450/500, Loss = 2.2305
Epoch 500/500, Loss = 1.9687


In [8]:
# 7. Find similar words

def most_similar(word, top_n=5):

    if word not in word_to_id:
        return []

    word_id = word_to_id[word]

    query_vector = embeddings[word_id].reshape(1, -1)

    similarities = cosine_similarity(
        query_vector,
        embeddings
    )[0]

    indices = np.argsort(similarities)[::-1]

    results = []

    for idx in indices:

        candidate = id_to_word[idx]

        if candidate == word:
            continue

        results.append(
            (candidate, similarities[idx])
        )

        if len(results) == top_n:
            break

    return results

In [9]:
# 8. Similar words to King

print("\nMost similar words to 'king':")

results = most_similar("king", 5)

for word, score in results:
    print(f"{word:15s} {score:.4f}")


Most similar words to 'king':
queen           0.9976
kingdom         0.9942
man             0.9774
the             0.9740
rules           0.9710


## b. Using Pretrained model (Gensim)

In [10]:
# Install gensim if necessary
!pip install -q gensim

import gensim.downloader as api

In [11]:
# Load pretrained GloVe embeddings

print("Loading pretrained GloVe model...")

glove_model = api.load("glove-wiki-gigaword-100")

print("Model loaded successfully!")

Loading pretrained GloVe model...
[==================================================] 100.0% 128.1/128.1MB downloaded
Model loaded successfully!


In [12]:
# Vector representation of King

king_vector = glove_model["king"]

print("\nEmbedding dimension:")
print(king_vector.shape)

print("\nFirst 10 dimensions of 'king':")
print(king_vector[:10])


Embedding dimension:
(100,)

First 10 dimensions of 'king':
[-0.32307 -0.87616  0.21977  0.25268  0.22976  0.7388  -0.37954 -0.35307
 -0.84369 -1.1113 ]


In [13]:
# Most similar words

print("\nMost similar words to 'king':")

similar_words = glove_model.most_similar(
    "king",
    topn=10
)

for word, similarity in similar_words:
    print(f"{word:15s} {similarity:.4f}")


Most similar words to 'king':
prince          0.7682
queen           0.7508
son             0.7021
brother         0.6986
monarch         0.6978
throne          0.6920
kingdom         0.6811
father          0.6802
emperor         0.6713
ii              0.6676


In [14]:
# Famous analogy
# king - man + woman ≈ queen

print("\nAnalogy: king - man + woman")

analogy = glove_model.most_similar(
    positive=["king", "woman"],
    negative=["man"],
    topn=10
)

for word, similarity in analogy:
    print(f"{word:15s} {similarity:.4f}")


Analogy: king - man + woman
queen           0.7699
monarch         0.6843
throne          0.6756
daughter        0.6595
princess        0.6521
prince          0.6517
elizabeth       0.6465
mother          0.6312
emperor         0.6106
wife            0.6099


## **Experiment 6.2:** To implement a statistical language model from scratch using the N-gram approach to predict the next word in a sequence based on the preceding words in a text corpus (shown below).

```
natural language processing is a branch of artificial intelligence.
natural language processing enables computers to understand human language.
machine learning is a branch of artificial intelligence.
deep learning is a part of machine learning.
language models learn patterns from text data.
a language model predicts the next word in a sentence.
natural language processing uses language models.
artificial intelligence includes machine learning and deep learning.
```

In [15]:
import re
from collections import defaultdict, Counter

In [16]:
# 1. Corpus given in Experiment 6

corpus = """
natural language processing is a branch of artificial intelligence.
natural language processing enables computers to understand human language.
machine learning is a branch of artificial intelligence.
deep learning is a part of machine learning.
language models learn patterns from text data.
a language model predicts the next word in a sentence.
natural language processing uses language models.
artificial intelligence includes machine learning and deep learning.
"""

In [17]:
# 2. Preprocessing

sentences = []

for sentence in corpus.strip().split("\n"):

    sentence = sentence.lower()

    words = re.findall(r"[a-z]+", sentence)

    sentences.append(words)

print("Processed sentences:")

for sentence in sentences:
    print(sentence)

Processed sentences:
['natural', 'language', 'processing', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence']
['natural', 'language', 'processing', 'enables', 'computers', 'to', 'understand', 'human', 'language']
['machine', 'learning', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence']
['deep', 'learning', 'is', 'a', 'part', 'of', 'machine', 'learning']
['language', 'models', 'learn', 'patterns', 'from', 'text', 'data']
['a', 'language', 'model', 'predicts', 'the', 'next', 'word', 'in', 'a', 'sentence']
['natural', 'language', 'processing', 'uses', 'language', 'models']
['artificial', 'intelligence', 'includes', 'machine', 'learning', 'and', 'deep', 'learning']


In [18]:
# 3. Build Unigram model

unigram = Counter()

for sentence in sentences:
    unigram.update(sentence)

print("\nMost common words:")
print(unigram.most_common(10))


Most common words:
[('language', 7), ('a', 5), ('learning', 5), ('natural', 3), ('processing', 3), ('is', 3), ('of', 3), ('artificial', 3), ('intelligence', 3), ('machine', 3)]


In [19]:
# 4. Build Bigram model

bigram_model = defaultdict(Counter)

for sentence in sentences:

    for i in range(len(sentence) - 1):

        current_word = sentence[i]
        next_word = sentence[i + 1]

        bigram_model[current_word][next_word] += 1

In [20]:
# 5. Build Trigram model

trigram_model = defaultdict(Counter)

for sentence in sentences:

    for i in range(len(sentence) - 2):

        context = (
            sentence[i],
            sentence[i + 1]
        )

        next_word = sentence[i + 2]

        trigram_model[context][next_word] += 1

In [21]:
# 6. Bigram prediction

def predict_bigram(word, top_n=5):

    word = word.lower()

    if word not in bigram_model:
        return []

    predictions = bigram_model[word]

    total = sum(predictions.values())

    results = []

    for next_word, count in predictions.most_common(top_n):

        probability = count / total

        results.append(
            (next_word, probability)
        )

    return results

In [22]:
# 7. Trigram prediction

def predict_trigram(word1, word2, top_n=5):

    context = (
        word1.lower(),
        word2.lower()
    )

    if context not in trigram_model:
        return []

    predictions = trigram_model[context]

    total = sum(predictions.values())

    results = []

    for next_word, count in predictions.most_common(top_n):

        probability = count / total

        results.append(
            (next_word, probability)
        )

    return results

In [23]:
# 8. Test Bigram predictions

print("\nBIGRAM PREDICTIONS")

test_words = [
    "natural",
    "machine",
    "deep",
    "artificial",
    "language"
]

for word in test_words:

    print(f"\nAfter '{word}':")

    predictions = predict_bigram(word)

    for next_word, probability in predictions:
        print(
            f"{next_word:15s} "
            f"Probability = {probability:.3f}"
        )


BIGRAM PREDICTIONS

After 'natural':
language        Probability = 1.000

After 'machine':
learning        Probability = 1.000

After 'deep':
learning        Probability = 1.000

After 'artificial':
intelligence    Probability = 1.000

After 'language':
processing      Probability = 0.500
models          Probability = 0.333
model           Probability = 0.167


In [24]:
# 9. Test Trigram predictions

print("\nTRIGRAM PREDICTIONS")

contexts = [
    ("natural", "language"),
    ("machine", "learning"),
    ("artificial", "intelligence"),
    ("language", "model")
]

for word1, word2 in contexts:

    print(
        f"\nAfter '{word1} {word2}':"
    )

    predictions = predict_trigram(
        word1,
        word2
    )

    for next_word, probability in predictions:
        print(
            f"{next_word:15s} "
            f"Probability = {probability:.3f}"
        )


TRIGRAM PREDICTIONS

After 'natural language':
processing      Probability = 1.000

After 'machine learning':
is              Probability = 0.500
and             Probability = 0.500

After 'artificial intelligence':
includes        Probability = 1.000

After 'language model':
predicts        Probability = 1.000


In [25]:
# 10. Interactive prediction

print("\nNEXT WORD PREDICTION")

user_input = input(
    "Enter one or two words: "
).lower().split()

if len(user_input) == 1:

    predictions = predict_bigram(
        user_input[0]
    )

elif len(user_input) >= 2:

    predictions = predict_trigram(
        user_input[-2],
        user_input[-1]
    )

else:
    predictions = []

if predictions:

    print("\nPredicted next words:")

    for word, probability in predictions:
        print(
            f"{word:15s} "
            f"{probability:.3f}"
        )

else:
    print("No prediction available.")


NEXT WORD PREDICTION
Enter one or two words: machine

Predicted next words:
learning        1.000
